# 0) Imports

In [1]:
import os
import sys
import glob
import json
import math
import time
import pdb
import pandas as pd
import numpy as np
import seaborn as sns
from tqdm import tqdm
from matplotlib import pyplot as plt
from pathlib import Path
from pathml.core import HESlide
from sklearn.utils import resample
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold

base = Path('/ix/rbao/Projects/panCancer_HE/')
results = base.joinpath('results')
scripts= base.joinpath('scripts')
sys.path.append(scripts.joinpath('pancancer_he_classifier'))
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2
data = base.joinpath('data')


ModuleNotFoundError: No module named 'pathml'

In [2]:
info.head()

,Unnamed: 0,slide,type,slide_path,annotation,anno_path
0,0,PHS18-17366_H&E.svs,Tumor,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...,PHS18-17366_H&E_anno.geojson,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...
1,1,PHS17-3558_H&E.svs,Tumor,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...,PHS17-3558_H&E_anno.geojson,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...
2,2,PHS17-29265_H&E.svs,Tumor,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...,PHS17-29265_H&E_anno.geojson,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...
3,3,PHS15-26213_H&E.svs,Tumor,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...,PHS15-26213_H&E_anno.geojson,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...
4,4,PHS13-14238_H&E.svs,Tumor,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...,PHS13-14238_H&E_anno.geojson,/ix/rbao/Projects/panCancer_HE/data/liver/aofe...


# 1) Generate training tile_df dataframe

## 1) Collect Liver training tiles

In [65]:
raw=data.joinpath('liver','aofei_liver_annotation')
sampleinfo = raw.joinpath('sampleinfo')
tiles= results.joinpath('liver_aofei','v9',
                        'tiles')
info = pd.read_csv(sampleinfo.joinpath('samples_24_slides.tsv'), 
                   sep='\t')

all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
version = 5 #Include case number
#Concatenate existing .tsv summaries:
tissue = 'liver'
all_tsv = pd.Series([str(x) for x in tiles.glob('*.tsv')])
missing = []
found = 0
for i,slide in enumerate(tqdm(info.slide.values)):
    slide=slide.split('.')[0]
    idx = all_tsv.str.contains(slide).values
    if any(idx):
        dat = pd.read_csv(all_tsv.loc[idx].values[0],sep='\t')
        tile_path = str(tiles.joinpath('224px',slide)) +'/'
        fns = tile_path + dat.tile 
        anno = dat.anno.copy()
        anno[anno.isna()] = 'notTumor'
        all_dat['fn'].extend(fns.values)
        all_dat['slide'].extend([i] * len(fns))
        all_dat['anno'].extend(anno.values)
        all_dat['case'].extend([slide.split('_')[0]] * len(fns))
        found += 1
    else:
        missing.append(slide)
n_missing = len(missing)
print('%d slides found, %d slides missing' % (found,n_missing))
print(missing)
    
tile_df = pd.DataFrame(all_dat)   
tile_df.loc[:,'tissue'] = tissue
tile_df.loc[:,'color_norm'] = True

fn = sampleinfo.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                tile_df.shape[0]))

print(fn)
tile_df.to_csv(fn,sep = '\t',index=False)
print(tile_df.shape)
tile_df.groupby('anno')['fn'].count()
tile_df.head(2)

100%|██████████| 24/24 [00:03<00:00,  7.83it/s]


14 slides found, 10 slides missing
['PHS17-29265_H&E', 'PHS13-14238_H&E', 'PHS11-40725_H&E', 'PHS21-22842_H&E', 'PHS17-18437_H&E', 'PHS19-7227_H&E', 'PHS19-22214_H&E', 'PHS17-20996_H&E', 'PHS17-24971_H&E', 'PHS19-34346_H&E']
/ix/rbao/Projects/panCancer_HE/data/liver/aofei_liver_annotation/sampleinfo/alltiles_df_v1_14samp_1019644tiles.tsv
(1019644, 6)


,fn,slide,case,anno,tissue,color_norm
0,/ix/rbao/Projects/panCancer_HE/results/liver_a...,0,PHS18-17366,notTumor,liver,True
1,/ix/rbao/Projects/panCancer_HE/results/liver_a...,0,PHS18-17366,notTumor,liver,True


In [51]:
tile_df.groupby(['case','anno'])['fn'].count()

case         anno    
PHS15-26213  Tumor        15354
             notTumor     16878
PHS17-28893  Tumor        11149
             notTumor     82233
PHS17-3558   Tumor        18604
             notTumor     29129
PHS18-17366  Tumor        16499
             notTumor    102633
PHS19-24159  Tumor        12691
             notTumor     60874
PHS20-33345  Tumor         8676
             notTumor     37551
PHS21-26778  Tumor         2189
             notTumor     52209
PHS21-27208  Tumor         1090
             notTumor     77342
PHS22-15513  Tumor         6104
             notTumor     47665
PHS22-16246  Tumor         8668
             notTumor     84596
PHS22-3190   Tumor         9028
             notTumor     63547
PVS21-13601  Tumor        40548
             notTumor     47855
PVS21-14620  Tumor         2709
             notTumor     71289
PVS22-8041   Tumor         4556
             notTumor     87978
Name: fn, dtype: int64

### Balance [WIP]

In [13]:
all_dat = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/liver/aofei_liver_annotation/sampleinfo/alltiles_df_v1_14samp_1019644tiles.tsv',
                      sep = '\t')


## 2) Collect colorectal training tiles

In [66]:
nct_crc_100k = data.joinpath('colorectal','NCT-CRC-HE-100K') #color normalized
colon_image = data.joinpath('colorectal','colon_image_sets') #all jpegs
print(nct_crc_100k.exists(), colon_image.exists())
all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
tissue='colorectal'
slide = np.nan
# all_fn = [x for x in nct_crc_100k.rglob('*.tif')]
for fn in tqdm(all_fn):
    if 'TUM' in fn.parent.parts[-1]:
        anno = 'Tumor'
    else:
        anno = 'notTumor'
    # if 'aca' in fn.parent.parts[-1]:
    #     anno = 'Tumor'
    # elif 'colon_n' in fn.parent.parts[-1]:
    #     anno = 'notTumor'
    all_dat['fn'].extend([str(fn)])
    all_dat['slide'].extend([''])
    all_dat['anno'].extend([anno])
    all_dat['case'].extend([''])

cr_df = pd.DataFrame(all_dat)                        
cr_df.loc[:,'tissue'] = tissue
cr_df.loc[:,'color_norm'] = True
fn = nct_crc_100k.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                cr_df.shape[0]))
print(fn)
cr_df.to_csv(fn,sep = '\t',index=False)
print(cr_df.shape)
cr_df.head(2)

True True


100%|██████████| 100000/100000 [00:00<00:00, 393905.70it/s]


/ix/rbao/Projects/panCancer_HE/data/colorectal/NCT-CRC-HE-100K/alltiles_df_v1_14samp_100000tiles.tsv
(100000, 6)


,fn,slide,case,anno,tissue,color_norm
0,/ix/rbao/Projects/panCancer_HE/data/colorectal...,,,notTumor,colorectal,True
1,/ix/rbao/Projects/panCancer_HE/data/colorectal...,,,notTumor,colorectal,True


In [67]:
cr_df.groupby('anno')['fn'].count()

anno
Tumor       14317
notTumor    85683
Name: fn, dtype: int64

## 3) Collect lung training tiles

In [69]:
lung_kaggle = data.joinpath('lung','lung_image_sets') #color normalized
print(lung_kaggle.exists())
all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
tissue='lung'
slide = np.nan
all_fn = [x for x in lung_kaggle.rglob('*.jpeg')]
for fn in tqdm(all_fn):
    if 'lung_n' in fn.parent.parts[-1]:
        anno = 'notTumor'
    else:
        anno = 'Tumor'
    all_dat['fn'].extend([str(fn)])
    all_dat['slide'].extend([''])
    all_dat['anno'].extend([anno])
    all_dat['case'].extend([''])

lu_df = pd.DataFrame(all_dat)                        
lu_df.loc[:,'tissue'] = tissue
lu_df.loc[:,'color_norm'] = False
fn = lung_kaggle.joinpath('alltiles_df_v1_%dtiles.tsv' % (lu_df.shape[0]))
print(fn)
lu_df.to_csv(fn,sep = '\t',index=False)
print(lu_df.shape)
lu_df.head(2)

True


100%|██████████| 15000/15000 [00:00<00:00, 292778.38it/s]


/ix/rbao/Projects/panCancer_HE/data/lung/lung_image_sets/alltiles_df_v1_14samp_15000tiles.tsv
(15000, 6)


,fn,slide,case,anno,tissue,color_norm
0,/ix/rbao/Projects/panCancer_HE/data/lung/lung_...,,,notTumor,lung,False
1,/ix/rbao/Projects/panCancer_HE/data/lung/lung_...,,,notTumor,lung,False


In [70]:
lu_df.groupby('anno')['fn'].count()

,fn,slide,case,tissue,color_norm
anno,,,,,
Tumor,10000,10000,10000,10000,10000
notTumor,5000,5000,5000,5000,5000


## 4) Collect skin / melanoma training tiles

In [7]:
raw=data.joinpath('skin','data','HE')
sampleinfo = data.joinpath('skin','sampleinfo')
tiles= results.joinpath('skin_rbao','v9',
                        'tiles')
info = pd.read_csv(sampleinfo.joinpath('samples_17_slides.tsv'), 
                   sep='\t')

all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
version = 5 #Include case number
#Concatenate existing .tsv summaries:
tissue = 'skin'
all_tsv = pd.Series([str(x) for x in tiles.glob('*.tsv')])
missing = []
found = 0
for i,slide in enumerate(tqdm(info.slide.values)):
    slide=slide.split('.')[0]
    idx = all_tsv.str.contains(slide).values
    if any(idx):
        dat = pd.read_csv(all_tsv.loc[idx].values[0],sep='\t')
        tile_path = str(tiles.joinpath('224px',slide)) +'/'
        fns = tile_path + dat.tile 
        anno = dat.anno.copy()
        anno[anno.isna()] = 'notTumor'
        all_dat['fn'].extend(fns.values)
        all_dat['slide'].extend([i] * len(fns))
        all_dat['anno'].extend(anno.values)
        all_dat['case'].extend([slide.split('_H&E')[0]] * len(fns))
        found += 1
    else:
        missing.append(slide)
n_missing = len(missing)
print('%d slides found, %d slides missing' % (found,n_missing))
print(missing)
    
tile_df = pd.DataFrame(all_dat)   
idx = tile_df.anno.values == 'anno'
tile_df.loc[idx,'anno'] = 'Tumor'
tile_df.loc[:,'tissue'] =  tissue
tile_df.loc[:,'color_norm'] = True

fn = sampleinfo.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                tile_df.shape[0]))

print(fn)
tile_df.to_csv(fn,sep = '\t',index=False)
print(tile_df.shape)
tile_df.groupby('anno')['fn'].count()
tile_df.head(2)

100%|██████████| 17/17 [00:00<00:00, 21.64it/s]


14 slides found, 3 slides missing
['MEL_PBC-PR-2885_1G_H&E', 'MEL_PBC-PR-2885_1D_H&E', 'MEL_PBC-PR-2861_H&E']
/ix/rbao/Projects/panCancer_HE/data/skin/sampleinfo/alltiles_df_v1_14samp_368033tiles.tsv
(368033, 6)


,fn,slide,case,anno,tissue,color_norm
0,/ix/rbao/Projects/panCancer_HE/results/skin_rb...,1,MEL_PBC-PR-2884_1I,notTumor,skin,True
1,/ix/rbao/Projects/panCancer_HE/results/skin_rb...,1,MEL_PBC-PR-2884_1I,notTumor,skin,True


In [6]:
sk_df.loc[sk_df.anno.values == 'anno','case'].unique()

array(['MEL_SMS18-10039', 'MEL_SYS10-577694', 'MEL_PD15-6806'],
      dtype=object)

## Last) Concatenate all together

In [8]:
sk_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/skin/sampleinfo/alltiles_df_v1_14samp_368033tiles.tsv', sep = '\t')
lv_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/liver/aofei_liver_annotation/sampleinfo/alltiles_df_v1_14samp_1019644tiles.tsv',sep = '\t')
lu_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/lung/lung_image_sets/alltiles_df_v1_14samp_15000tiles.tsv',sep='\t')
cr_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/colorectal/NCT-CRC-HE-100K/alltiles_df_v1_14samp_100000tiles.tsv', sep = '\t')

In [9]:
all_df = pd.concat((sk_df,lv_df,lu_df,cr_df),axis=0)
fn = data.joinpath('skin_lung_liver_colo_df_v1_%dtiles.tsv' % (all_df.shape[0]))
print(fn)
all_df.to_csv(fn,sep = '\t')
print('Finished')

/ix/rbao/Projects/panCancer_HE/data/skin_lung_liver_colo_df_v1_1502677tiles.tsv
Finished


In [2]:
all_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/skin_lung_liver_colo_df_v1_1502677tiles.tsv',
                     sep = '\t')

/scratch/slurm-2140842/ipykernel_55026/3248736025.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/skin_lung_liver_colo_df_v1_1354174tiles.tsv',


In [10]:
all_df.groupby(['tissue','anno'])['fn'].count()

tissue      anno    
colorectal  Tumor        14317
            notTumor     85683
liver       Tumor       157865
            notTumor    861779
lung        Tumor        10000
            notTumor      5000
skin        Tumor        15453
            notTumor    352580
Name: fn, dtype: int64

In [16]:
all_df.case.unique()

array(['MEL_PBC-PR-2863', 'MEL_JMS19-1718', 'MEL_PBC-PR-1518',
       'MEL_PBC-PR-2875', 'MEL_PBC-PR-2871', 'MEL_PBC-PR-2870',
       'MEL_5216', 'PHS18-17366', 'PHS17-3558', 'PHS15-26213',
       'PHS21-27208', 'PHS20-33345', 'PHS22-3190', 'PVS21-13601',
       'PHS17-28893', 'PVS21-14620', 'PHS22-15513', 'PHS21-26778',
       'PVS22-8041', 'PHS22-16246', 'PHS19-24159', nan], dtype=object)

In [8]:
all_df.groupby(['anno'])['fn'].count()

anno
Tumor       182182
notTumor    952462
Name: fn, dtype: int64

# 2) Create balanced tile set:

In [52]:
all_df = pd.read_csv('/ix/rbao/Projects/panCancer_HE/data/skin_lung_liver_colo_df_v1_1354174tiles.tsv',
                     sep = '\t',
                    low_memory = False)
print('Finished')

Finished


In [11]:
n_tumor_per_tissue = 14000
state = 36
collect = pd.DataFrame([])
for i,tissue in enumerate(all_df.tissue.unique()):
    idx = all_df.tissue == tissue
    cases = all_df.loc[idx,'case'].unique()
    #If cases are present, balance by cases
    if isinstance(cases[0],str):
        n_per_case = n_tumor_per_tissue // len(cases)
        print(tissue,n_per_case)
        for case in cases:
            idx2 = all_df.case.values == case
            keep = all_df.loc[idx & idx2,:].copy()
            t = keep.anno == 'Tumor'
            n = keep.anno == 'notTumor'
            if np.sum(t) < n_per_case:
                replace = True
            else:
                replace = False
            keep_t = resample(keep.loc[t,:],
                        n_samples=n_per_case,
                        random_state=state, 
                        replace=replace, )
            if np.sum(n) < n_per_case:
                replace = True
            else:
                replace = False
            keep_n = resample(keep.loc[n,:],
                       n_samples=n_per_case,
                        random_state=state, 
                        replace=replace, )
            collect = pd.concat((collect,keep_t,keep_n),axis=0)
    #If cases are not present, do not balance by case:
    else:
        print(tissue,n_tumor_per_tissue)
        keep = all_df.loc[idx,:].copy()
        t = keep.anno == 'Tumor'
        if np.sum(t) < n_tumor_per_tissue:
                replace = True
        else:
                replace = False
        n = keep.anno == 'notTumor'

        keep_t = resample(keep.loc[t,:],
                n_samples=n_tumor_per_tissue,
                random_state=state, 
                replace=replace, )
        if np.sum(n) < n_tumor_per_tissue:
                replace = True
        else:
                replace = False
        keep_n = resample(keep.loc[n,:],
                   n_samples=n_tumor_per_tissue,
                    random_state=state, 
                    replace=replace, )
        collect = pd.concat((collect,keep_t,keep_n),axis=0)
collect = collect.reset_index(drop=True)
fn = data.joinpath('balanced_%d_sk_lu_lv_cr_df_v1_%dtiles.tsv' % (n_tumor_per_tissue,collect.shape[0]))
print(fn)
collect.to_csv(fn,sep = '\t')
print('Finished')
collect.groupby(['tissue','anno'])['fn'].count()

skin 714
liver 714
lung 10000
colorectal 10000
/ix/rbao/Projects/panCancer_HE/data/balanced_10000_sk_lu_lv_cr_df_v1_79984tiles.tsv
Finished


tissue      anno    
colorectal  Tumor       10000
            notTumor    10000
liver       Tumor        9996
            notTumor     9996
lung        Tumor       10000
            notTumor    10000
skin        Tumor        9996
            notTumor     9996
Name: fn, dtype: int64

## Test splitter for train on 3 cancer, test on 1 other:

In [64]:
kfold = 4
splitter = GroupKFold(n_splits=kfold)
for slide_train_idx, slide_test_idx in splitter.split(collect.loc[:, 'fn'],
                                                      collect.loc[:, 'anno'],
                                                      groups=collect.loc[:, 'tissue']):
    print(collect.loc[slide_train_idx,:].groupby(['tissue','anno'])['fn'].count())

tissue      anno    
colorectal  Tumor       5000
            notTumor    5000
liver       Tumor       4998
            notTumor    4998
skin        Tumor       4998
            notTumor    4998
Name: fn, dtype: int64
tissue  anno    
liver   Tumor       4998
        notTumor    4998
lung    Tumor       5000
        notTumor    5000
skin    Tumor       4998
        notTumor    4998
Name: fn, dtype: int64
tissue      anno    
colorectal  Tumor       5000
            notTumor    5000
liver       Tumor       4998
            notTumor    4998
lung        Tumor       5000
            notTumor    5000
Name: fn, dtype: int64
tissue      anno    
colorectal  Tumor       5000
            notTumor    5000
lung        Tumor       5000
            notTumor    5000
skin        Tumor       4998
            notTumor    4998
Name: fn, dtype: int64


## Test splitter for: Train on all cancer, test on all cancer

In [66]:
kfold = 10
reps = 1
i = 1
splitter = RepeatedKFold(n_splits = kfold,
                         n_repeats = reps,
                         random_state = state)
for slide_train_idx, slide_test_idx in splitter.split(collect.loc[:, 'fn'],
                                                      collect.loc[:, 'anno'],
                                                      ):
    print(i)
    print(collect.loc[slide_train_idx,:].groupby(['tissue','anno'])['fn'].count())
    i += 1

1
tissue      anno    
colorectal  Tumor       4494
            notTumor    4481
liver       Tumor       4499
            notTumor    4505
lung        Tumor       4488
            notTumor    4532
skin        Tumor       4478
            notTumor    4515
Name: fn, dtype: int64
2
tissue      anno    
colorectal  Tumor       4489
            notTumor    4499
liver       Tumor       4502
            notTumor    4471
lung        Tumor       4496
            notTumor    4491
skin        Tumor       4518
            notTumor    4526
Name: fn, dtype: int64
3
tissue      anno    
colorectal  Tumor       4483
            notTumor    4492
liver       Tumor       4492
            notTumor    4495
lung        Tumor       4518
            notTumor    4494
skin        Tumor       4529
            notTumor    4490
Name: fn, dtype: int64
4
tissue      anno    
colorectal  Tumor       4492
            notTumor    4520
liver       Tumor       4502
            notTumor    4494
lung        Tumor       451